In [ ]:
(* load the setup of the notebooks *)
#use "../setup.ml";;

# Map Coloring

This short notebook shows how *map coloring* can be formulated as a *constraint satisfaction problem*.
In map coloring, the goal is to color the states shown in a given map such that no two bordering states
share the same color.  As an example, consider the map of Australia that is shown below.  Australia
has seven different states:
* [Western Australia](https://en.wikipedia.org/wiki/Western_Australia),
* [Northern Territory](https://en.wikipedia.org/wiki/Northern_Territory),
* [South Australia](https://en.wikipedia.org/wiki/South_Australia),
* [Queensland](https://en.wikipedia.org/wiki/Queensland),
* [New South Wales](https://en.wikipedia.org/wiki/New_South_Wales),
* [Victoria](https://en.wikipedia.org/wiki/Victoria_(Australia)), and
* [Tasmania](https://en.wikipedia.org/wiki/Tasmania).

As Tasmania is an island that does not share a border with any of the other states, it can be colored with any given color.

<img src="australia.png" alt="map of australia" width="500">

We import the constraint solver from the notebook `02-Backtracking-Constraint-Solver.ipynb`.

In [ ]:
import_notebook "02-Backtracking-Constraint-Solver.ipynb"

The variables of a CSP are written in the constraints as OCaml identifiers, which have to start with a lower case letter.  Therefore, we use the variables `wa`, `nsw`, `t`, `v`, `nt`, `sa`, and `q` for the states.

The backtracking constraint solver requires integer values, so we map our colors to integers.  The association list `color_map` maps every value to the name of its color.

In [ ]:
let color_map : (int * string) list = [(1, "red"); (2, "blue"); (3, "green")]

The function `map_coloring_csp` takes the argument `()`.  It returns a CSP that encodes the *map coloring problem* for Australia.  The constraint `"wa <> nt"` states that Western Australia and the Northern Territory have different colors.

In [ ]:
let map_coloring_csp () : csp = {
  variables   = ["wa"; "nsw"; "t"; "v"; "nt"; "sa"; "q"];
  values      = [1; 2; 3];   (* 1: red, 2: blue, 3: green *)
  constraints = [
    "wa <> nt"; "wa <> sa";
    "nt <> sa"; "nt <> q";
    "sa <> q";  "sa <> nsw"; "sa <> v";
    "q <> nsw"; "nsw <> v"
  ]
}

In [ ]:
let csp = map_coloring_csp ()

In [ ]:
let solution = solve csp

As there are $3$ different values and $7$ different variables, there are $3^7 = 2187$ different ways to assign values to the variables.  In OCaml, the operator `**` computes powers of floating point numbers.

In [ ]:
3. ** 7.

## Visualization

The association list `australia_paths` maps every state to an SVG path that describes its outline.  A label consists of the position of its center and the lines of its text.  The association list `labels` maps every state to its label.

In [ ]:
let australia_paths : (string * string) list = [
  ("wa",  "M 190,340 L 180,310 L 130,280 L 60,250 L 40,200 L 60,150 L 90,120 L 150,80 L 240,70 L 240,250 L 210,340 Z");
  ("nt",  "M 240,70 L 330,70 L 340,100 L 330,200 L 240,200 Z");
  ("sa",  "M 240,200 L 330,200 L 330,250 L 360,270 L 350,340 L 290,340 L 260,310 L 210,340 L 240,250 Z");
  ("q",   "M 330,70 L 360,20 L 400,60 L 430,150 L 450,200 L 450,250 L 330,250 L 330,200 Z");
  ("nsw", "M 330,250 L 450,250 L 440,320 L 350,320 L 350,270 Z");
  ("v",   "M 350,320 L 440,320 L 420,360 L 360,360 Z");
  ("t",   "M 390,390 L 420,390 L 410,430 L 380,430 Z")
]

type label = { x : int; y : int; lines : string list }

let labels : (string * label) list = [
  ("wa",  { x = 140; y = 190; lines = ["Western"; "Australia"] });
  ("nt",  { x = 285; y = 130; lines = ["Northern"; "Territory"] });
  ("sa",  { x = 290; y = 265; lines = ["South"; "Australia"] });
  ("q",   { x = 390; y = 140; lines = ["Queensland"] });
  ("nsw", { x = 395; y = 280; lines = ["New South"; "Wales"] });
  ("v",   { x = 395; y = 340; lines = ["Victoria"] });
  ("t",   { x = 400; y = 415; lines = ["Tasmania"] })
]

In order to display HTML in the notebook, we load the library `jupyter.notebook`, which provides the function `Jupyter_notebook.display`.

In [ ]:
#require "jupyter.notebook";;

The function `show_solution` takes one argument:
 * `assignment` is a partial variable assignment, i.e. a map that maps
   some of the variables to the numbers of their colors.

It displays the map of Australia, where every state is filled with its color.  States without a color are shown in light gray.

**Implementation:** For every state, `RMap.find_opt` returns the number of its color, if any.  `Option.map f o` applies `f` to the value inside `o`, and `Option.value o ~default:d` returns this value or `d`, if `o` is `None`.  The SVG code consists of a path for every state and a text for every label.

In [ ]:
let show_solution (assignment : assignment) : unit =
  let colors = [("red", "#ff8787"); ("green", "#69db7c"); ("blue", "#4dabf7")] in
  let fill state =
    RMap.find_opt state assignment
    |> Option.map (fun value -> List.assoc (List.assoc value color_map) colors)
    |> Option.value ~default:"#f1f3f5"
  in
  let svg_paths =
    String.concat ""
      (List.map
         (fun (state, path) ->
            Printf.sprintf "<path d=\"%s\" fill=\"%s\" stroke=\"#888\" stroke-width=\"1\" />" path (fill state))
         australia_paths)
  in
  let svg_labels =
    String.concat ""
      (List.map
         (fun (_, l) ->
            let tspans =
              String.concat ""
                (List.mapi
                   (fun i line -> Printf.sprintf "<tspan x=\"%d\" dy=\"%s\">%s</tspan>" l.x (if i = 0 then "0" else "1.2em") line)
                   l.lines)
            in
            Printf.sprintf "<text x=\"%d\" y=\"%d\" font-family=\"Arial, sans-serif\" font-size=\"12\" font-weight=\"bold\" fill=\"#343a40\" text-anchor=\"middle\" pointer-events=\"none\">%s</text>"
              l.x l.y tspans)
         labels)
  in
  let html =
    Printf.sprintf "<div style=\"border: 1px solid #dee2e6; padding: 10px; border-radius: 5px; display: inline-block; background: white;\"><svg width=\"500\" height=\"450\" viewBox=\"0 0 500 450\">%s%s</svg></div>"
      svg_paths svg_labels
  in
  ignore (Jupyter_notebook.display "text/html" html)

As `solution` has the type `assignment option`, we have to distinguish the two cases.

In [ ]:
match solution with
| Some s -> show_solution s
| None   -> print_endline "No solution found."

Below, we have changed the ordering of the variables.  With this new ordering, the constraint solver never needs to backtrack.

In [ ]:
let map_coloring_csp2 () : csp = {
  variables   = ["wa"; "nt"; "sa"; "q"; "nsw"; "v"; "t"];
  values      = [1; 2; 3];
  constraints = [
    "wa <> nt"; "wa <> sa";
    "nt <> sa"; "nt <> q";
    "sa <> q";  "sa <> nsw"; "sa <> v";
    "q <> nsw"; "nsw <> v"
  ]
}

In [ ]:
let csp_optimized = map_coloring_csp2 ();;
match solve csp_optimized with
| Some s -> show_solution s
| None   -> print_endline "No solution found."